# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta1805/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Warehouse connection ready.")

Warehouse connection ready.


In [6]:
queue_data = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
LIMIT 10000
""").df()

print("Queue rows:", len(queue_data))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue rows: 10000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

Prioritize content that ranks in positions 4–20 and has higher search visibility, because these pages have both room for improvement and a meaningful search opportunity.

Pages ranking in positions 1–3 receive lower priority because they already have strong search visibility. Pages ranking 21+ receive a smaller position score because their low visibility makes the opportunity less certain.

GSC impressions are used as an impression-volume proxy. Higher impression volume increases the opportunity score but is not treated as evidence of poor performance by itself.

The rule uses only current-period observed fields. It does not use future performance or a future decline label.

### Reason codes

- `POSITION_VOLUME_OPPORTUNITY` — the page has a meaningful position opportunity and/or impression volume.
- `LOW_OPPORTUNITY` — the page has limited evidence of an actionable opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Build the baseline action score

import os
import pandas as pd

queue = queue_data.copy()

# Remove invalid search positions
queue = queue[queue["gsc_avg_position"] > 0].copy()

# Create impression-volume buckets using the same thresholds
q1 = queue["gsc_impressions"].quantile(0.33)
q2 = queue["gsc_impressions"].quantile(0.66)

queue["volume_bucket"] = pd.cut(
    queue["gsc_impressions"],
    bins=[-float("inf"), q1, q2, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

# Position score
def position_score(position):
    if position <= 3:
        return 0
    elif position <= 10:
        return 2
    elif position <= 20:
        return 3
    else:
        return 1

queue["position_score"] = queue["gsc_avg_position"].apply(position_score)

# Volume score
volume_points = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

queue["volume_score"] = (
    queue["volume_bucket"]
    .map(volume_points)
    .astype(int)
)

# Final baseline score
queue["baseline_score"] = (
    queue["position_score"] +
    queue["volume_score"]
)

# Action label
def action_label(score):
    if score >= 4:
        return "Refresh"
    elif score >= 2:
        return "Monitor"
    else:
        return "Low priority"

queue["action"] = queue["baseline_score"].apply(action_label)

# Reason code
queue["reason_code"] = queue["action"].map({
    "Refresh": "POSITION_VOLUME_OPPORTUNITY",
    "Monitor": "POSITION_VOLUME_OPPORTUNITY",
    "Low priority": "LOW_OPPORTUNITY"
})

# Rank highest-priority items first
queue = queue.sort_values(
    ["baseline_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

# Select final output columns
baseline_queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "volume_bucket",
        "baseline_score",
        "reason_code",
        "action"
    ]
].copy()

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Write required CSV
output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print("Queue rows:", len(baseline_queue))
print("CSV written to:", output_path)

baseline_queue.head(10)

Queue rows: 9337
CSV written to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,volume_bucket,baseline_score,reason_code,action
0,1,client_65de48885f4ef01b,content_79a53bf595360d49,2026-03-01,347,0,10.095101,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
1,2,client_73cda7b4e4f265ea,content_6cec704c446e1472,2026-03-01,308,0,10.084416,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
2,3,client_c182d11e4862a37d,content_f4a0e5c90b283626,2026-03-01,250,0,19.448000,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
3,4,client_73cda7b4e4f265ea,content_499c5ab71d3828de,2026-03-01,222,0,18.563063,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
4,5,client_73cda7b4e4f265ea,content_338dd9be259c61b9,2026-03-01,215,1,11.167442,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
5,6,client_73cda7b4e4f265ea,content_4a8683d3598718fa,2026-03-01,172,0,15.534884,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
6,7,client_c182d11e4862a37d,content_a5403416f0d9ddef,2026-03-01,160,0,15.106250,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
7,8,client_73cda7b4e4f265ea,content_1d9f00db9317d029,2026-03-01,150,0,10.600000,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
8,9,client_73cda7b4e4f265ea,content_9675e1a970ce76c9,2026-03-01,144,0,10.986111,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh
9,10,client_c182d11e4862a37d,content_da76e1818babb4ce,2026-03-01,132,0,11.348485,High,5,POSITION_VOLUME_OPPORTUNITY,Refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The top 20 items are reviewed individually rather than treating the baseline score as ground truth.

The confidence note describes why the item received its ranking under the current rule.

The “what would make it wrong” note identifies evidence that could invalidate the action, such as low-quality impressions, unusual query mix, temporary ranking behavior, or missing context not captured by the baseline features.

In [10]:
top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    position = row["gsc_avg_position"]
    impressions = row["gsc_impressions"]

    return (
        f"Refresh because position {position:.1f} is in the 4–20 opportunity range "
        f"and impressions ({impressions}) are in the {row['volume_bucket']} volume bucket."
    )

def wrong_reason(row):
    return (
        "Could be wrong if the impressions are driven by low-value queries, "
        "the ranking is temporary, or the page has context not captured by this baseline."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_reason, axis=1)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

top20_review

,rank,content_hash_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_79a53bf595360d49,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 10.1 is in the 4–20 o...,Could be wrong if the impressions are driven b...
1,2,content_6cec704c446e1472,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 10.1 is in the 4–20 o...,Could be wrong if the impressions are driven b...
2,3,content_f4a0e5c90b283626,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 19.4 is in the 4–20 o...,Could be wrong if the impressions are driven b...
3,4,content_499c5ab71d3828de,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 18.6 is in the 4–20 o...,Could be wrong if the impressions are driven b...
4,5,content_338dd9be259c61b9,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 11.2 is in the 4–20 o...,Could be wrong if the impressions are driven b...
5,6,content_4a8683d3598718fa,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 15.5 is in the 4–20 o...,Could be wrong if the impressions are driven b...
6,7,content_a5403416f0d9ddef,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 15.1 is in the 4–20 o...,Could be wrong if the impressions are driven b...
7,8,content_1d9f00db9317d029,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 10.6 is in the 4–20 o...,Could be wrong if the impressions are driven b...
8,9,content_9675e1a970ce76c9,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 11.0 is in the 4–20 o...,Could be wrong if the impressions are driven b...
9,10,content_da76e1818babb4ce,Refresh,POSITION_VOLUME_OPPORTUNITY,5,Refresh because position 11.3 is in the 4–20 o...,Could be wrong if the impressions are driven b...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline is intentionally simple, so some ranked items may be weak recommendations.

Potential weak picks include pages where high impression volume does not correspond to a clear improvement opportunity, or where the observed search position may be temporary.

The baseline should therefore be treated as a prioritization queue, not as proof that a page needs a refresh.

### Leakage check

The baseline uses only current-period observed fields:
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`

It does not use:
- future-period performance
- decline labels
- future impressions or clicks
- product-generated flags
- target/label-derived fields
- client or content IDs as predictive signals

In [11]:
print("=== Weak-pick check ===")

weak_picks = baseline_queue[
    baseline_queue["baseline_score"] <= 2
].head(10)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "gsc_avg_position",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

print("\n=== Leakage check ===")

scoring_inputs = [
    "gsc_impressions",
    "gsc_avg_position"
]

print("Scoring inputs:", scoring_inputs)

excluded_fields = [
    "future-period performance",
    "decline label",
    "future impressions",
    "future clicks",
    "product-generated flags"
]

print("Excluded future/label-derived inputs:", excluded_fields)

print("\nLeakage check: PASS")

=== Weak-pick check ===


,rank,content_hash_id,gsc_impressions,gsc_avg_position,baseline_score,reason_code,action
4491,4492,content_29c4a3831609805d,5932,1.938975,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4492,4493,content_91e1b34e81e54cad,3034,2.500989,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4493,4494,content_e241d6415ac9e534,2767,2.579689,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4494,4495,content_ac744f16f3d99a5b,2622,2.886346,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4495,4496,content_3508adb0f05ec0b9,2412,2.861526,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4496,4497,content_b720d2b38730e3cc,1585,2.588013,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4497,4498,content_3fd56bc37f6ac016,1547,2.991597,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4498,4499,content_76f9b5358889447c,1259,2.003177,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4499,4500,content_35ea4a32a987be2d,1252,1.612620,2,POSITION_VOLUME_OPPORTUNITY,Monitor
4500,4501,content_be57e90645e0247d,1225,2.862857,2,POSITION_VOLUME_OPPORTUNITY,Monitor



=== Leakage check ===
Scoring inputs: ['gsc_impressions', 'gsc_avg_position']
Excluded future/label-derived inputs: ['future-period performance', 'decline label', 'future impressions', 'future clicks', 'product-generated flags']

Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.